In [ ]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS emergency_data
OPTIONS(location="us");

Query is running:   0%|          |

""


In [ ]:
%%bigquery
LOAD DATA OVERWRITE emergency_data.response_times
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv'],
  skip_leading_rows = 1
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT * FROM emergency_data.response_times LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


In [ ]:
%%bigquery
CREATE OR REPLACE MODEL emergency_data.response_model
OPTIONS (
  model_type = 'linear_reg',
  input_label_cols = ['response_time']
) AS
SELECT
  *
FROM
  `emergency_data.response_times`
WHERE
  response_time IS NOT NULL;

Query is running:   0%|          |

""


In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.EVALUATE(MODEL emergency_data.response_model);

Query is running:   0%|          |

Downloading:   0%|          |

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.742121,4.770665,0.014883,1.474358,0.829923,0.829955


In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.PREDICT(MODEL `emergency_data.response_model`,
    (
    SELECT
      'Medical' AS call_type,
      'High' AS priority,
      42.3601 AS latitude,
      -71.0589 AS longitude,
      14 AS hour_of_day
    UNION ALL
    SELECT
      'Fire' AS call_type,
      'Medium' AS priority,
      42.3500 AS latitude,
      -71.1000 AS longitude,
      2 AS hour_of_day
    )
  );

Executing query with job ID: 64fdafa9-8d72-48aa-8c5e-2ec0162a1e0f
Query executing: 0.31s


ERROR:
 400 Invalid table-valued function ML.PREDICT
Column call_id is not found in the input data to the PREDICT function. at [4:3]; reason: invalidQuery, location: query, message: Invalid table-valued function ML.PREDICT
Column call_id is not found in the input data to the PREDICT function. at [4:3]

Location: US
Job ID: 64fdafa9-8d72-48aa-8c5e-2ec0162a1e0f



In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.PREDICT(MODEL `emergency_data.response_model`,
    (
    SELECT
      101 AS call_id,  -- Added dummy call_id
      'Medical' AS call_type,
      'High' AS priority,
      42.3601 AS latitude,
      -71.0589 AS longitude,
      14 AS hour_of_day
    UNION ALL
    SELECT
      102 AS call_id,  -- Added dummy call_id
      'Fire' AS call_type,
      'Medium' AS priority,
      42.3500 AS latitude,
      -71.1000 AS longitude,
      2 AS hour_of_day
    )
  );

Executing query with job ID: 8df3db32-c116-4a34-9c91-85cacf2a2b95
Query executing: 0.34s


ERROR:
 400 Invalid table-valued function ML.PREDICT
Column call_timestamp is not found in the input data to the PREDICT function. at [4:3]; reason: invalidQuery, location: query, message: Invalid table-valued function ML.PREDICT
Column call_timestamp is not found in the input data to the PREDICT function. at [4:3]

Location: US
Job ID: 8df3db32-c116-4a34-9c91-85cacf2a2b95



In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.PREDICT(MODEL `emergency_data.response_model`,
    (
    SELECT
      101 AS call_id,
      CURRENT_TIMESTAMP() AS call_timestamp,
      'Medical' AS call_type,
      'High' AS priority,
      42.3601 AS latitude,
      -71.0589 AS longitude,
      14 AS hour_of_day
    )
  );

Executing query with job ID: deb09930-1f82-46f0-9bde-ad9b26813f81
Query executing: 0.30s


ERROR:
 400 Invalid table-valued function ML.PREDICT
Column location is not found in the input data to the PREDICT function. at [4:3]; reason: invalidQuery, location: query, message: Invalid table-valued function ML.PREDICT
Column location is not found in the input data to the PREDICT function. at [4:3]

Location: US
Job ID: deb09930-1f82-46f0-9bde-ad9b26813f81



In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.PREDICT(MODEL `emergency_data.response_model`,
    (
    SELECT
      * REPLACE(
        'Medical' AS call_type,
        'High' AS priority,
        14 AS hour_of_day
      )
    FROM `emergency_data.response_times`
    LIMIT 1
    )
  );

Executing query with job ID: 6f28ad5f-288c-421d-8b60-ed0e7b34d8c1
Query executing: 0.37s


ERROR:
 400 Column priority in SELECT * REPLACE list does not exist at [9:19]; reason: invalidQuery, location: query, message: Column priority in SELECT * REPLACE list does not exist at [9:19]

Location: US
Job ID: 6f28ad5f-288c-421d-8b60-ed0e7b34d8c1



In [ ]:
%%bigquery
SELECT
  *
FROM
  ML.PREDICT(MODEL `emergency_data.response_model`,
    (
    SELECT
      * EXCEPT(response_time) -- Remove the actual answer so the model can predict it
    FROM `emergency_data.response_times`
    LIMIT 1
    )
  );

Query is running:   0%|          |

Downloading:   0%|          |

,predicted_response_time,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available
0,24.371718,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3
